In [45]:
import os
from dotenv import load_dotenv
from urllib.request import urlopen
import json
import pandas as pd
from google.cloud import bigquery
from urllib.error import HTTPError
from datetime import datetime
import time
from io import BytesIO
from google.cloud import storage

load_dotenv()

True

In [37]:
BASE_URL = os.getenv("BASE_URL", "").strip("/")
YEAR = int(os.getenv("YEAR", ""))
URL = f"{BASE_URL}/session_result?session_key="
GCS_BUCKET = os.getenv("GCS_BUCKET")
GCS_MEDAL_PUSH = os.getenv("GCS_MEDAL_PUSH")
BLOB_PATH = f"{GCS_MEDAL_PUSH}/season={YEAR}/session_results/2026.parquet"
PROJECT = os.getenv("PROJECT", "").strip()
LOCATION = os.getenv("LOCATION", "").strip()
DATASET_FETCH = f"{PROJECT}.openf1_staging.stg_sessions"

In [38]:
client = bigquery.Client(project=PROJECT, location=LOCATION)
sql = f"""
    SELECT 
        *
    FROM {DATASET_FETCH}
"""
df_sessions = client.query(sql).to_dataframe()


c:\Users\USER\anaconda3\envs\de\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [39]:
df_used = df_sessions[["session_key", "date_start", "session_type", "session_name", "year"]]
now = pd.Timestamp.now(tz="UTC")
df_used = df_used.loc[
    df_used["date_start"] < now
]

In [40]:
frames = []
for s in df_used["session_key"]:
    url = f"{URL}{int(s)}"
    print(f"Fetching for {url}")
    while True:
        try:
            with urlopen(url, timeout=120) as resp:
                raw = resp.read().decode("utf-8")
            data = json.loads(raw)
            break
        except HTTPError as e:
            if e.code == 429:
                now = datetime.now()
                seconds_into_minute = now.second + now.microsecond / 1_000_000
                sleep_secs = 60 - seconds_into_minute
                if sleep_secs <= 0:
                    sleep_secs = 60
                print(f"{e.code}: Waiting {sleep_secs:.1f}s until next minute...")
                time.sleep(sleep_secs)
                continue 
            if e.code == 404:
                print(f"{e.code}: Not found for {url}")
                data = None
                break
            raise 
    if not data:
        continue
    tmp = pd.DataFrame(data)
    tmp["ingested_at"] = pd.Timestamp.now(tz="UTC")
    frames.append(tmp)
df_sr = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


Fetching for https://api.openf1.org/v1/session_result?session_key=11465
Fetching for https://api.openf1.org/v1/session_result?session_key=11466
Fetching for https://api.openf1.org/v1/session_result?session_key=11467
Fetching for https://api.openf1.org/v1/session_result?session_key=11470
Fetching for https://api.openf1.org/v1/session_result?session_key=11469
Fetching for https://api.openf1.org/v1/session_result?session_key=11468
Fetching for https://api.openf1.org/v1/session_result?session_key=11227
Fetching for https://api.openf1.org/v1/session_result?session_key=11228
Fetching for https://api.openf1.org/v1/session_result?session_key=11229
Fetching for https://api.openf1.org/v1/session_result?session_key=11230
Fetching for https://api.openf1.org/v1/session_result?session_key=11234
Fetching for https://api.openf1.org/v1/session_result?session_key=11235
Fetching for https://api.openf1.org/v1/session_result?session_key=11236
Fetching for https://api.openf1.org/v1/session_result?session_ke

In [41]:
df_used

,session_key,date_start,session_type,session_name,year
0,11465,2026-02-11 07:00:00+00:00,Practice,Day 1,2026
1,11466,2026-02-12 07:00:00+00:00,Practice,Day 2,2026
2,11467,2026-02-13 07:00:00+00:00,Practice,Day 3,2026
3,11470,2026-02-18 07:00:00+00:00,Practice,Day 1,2026
4,11469,2026-02-19 07:00:00+00:00,Practice,Day 2,2026
5,11468,2026-02-20 07:00:00+00:00,Practice,Day 3,2026
6,11227,2026-03-06 01:30:00+00:00,Practice,Practice 1,2026
7,11228,2026-03-06 05:00:00+00:00,Practice,Practice 2,2026
8,11229,2026-03-07 01:30:00+00:00,Practice,Practice 3,2026
9,11230,2026-03-07 05:00:00+00:00,Qualifying,Qualifying,2026


In [42]:
df_sr.loc[~df_sr["points"].isna()].sort_values(
    by=["session_key"],
    ascending=True
)

df_sr = pd.merge(left=df_sr, right=df_used, on="session_key")

In [43]:
df_sr.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 673 entries, 0 to 672
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype              
---  ------          --------------  -----              
 0   position        611 non-null    float64            
 1   driver_number   673 non-null    int64              
 2   number_of_laps  672 non-null    float64            
 3   dnf             673 non-null    bool               
 4   dns             673 non-null    bool               
 5   dsq             673 non-null    bool               
 6   gap_to_leader   609 non-null    object             
 7   meeting_key     673 non-null    int64              
 8   session_key     673 non-null    int64              
 9   duration        562 non-null    object             
 10  ingested_at     673 non-null    datetime64[us, UTC]
 11  points          176 non-null    float64            
 12  date_start      673 non-null    datetime64[us, UTC]
 13  session_type    673 non-null    obj

In [47]:
df_sr = df_sr.drop(columns=["gap_to_leader"])

In [44]:
df_sr

,position,driver_number,number_of_laps,dnf,dns,dsq,gap_to_leader,meeting_key,session_key,duration,ingested_at,points,date_start,session_type,session_name,year
0,1.0,1,58.0,False,False,False,0.0,1304,11465,94.669,2026-06-01 14:01:53.362578+00:00,NaN,2026-02-11 07:00:00+00:00,Practice,Day 1,2026
1,2.0,3,136.0,False,False,False,0.129,1304,11465,94.798,2026-06-01 14:01:53.362578+00:00,NaN,2026-02-11 07:00:00+00:00,Practice,Day 1,2026
2,3.0,16,80.0,False,False,False,0.521,1304,11465,95.19,2026-06-01 14:01:53.362578+00:00,NaN,2026-02-11 07:00:00+00:00,Practice,Day 1,2026
3,4.0,31,115.0,False,False,False,0.909,1304,11465,95.578,2026-06-01 14:01:53.362578+00:00,NaN,2026-02-11 07:00:00+00:00,Practice,Day 1,2026
4,5.0,81,54.0,False,False,False,0.933,1304,11465,95.602,2026-06-01 14:01:53.362578+00:00,NaN,2026-02-11 07:00:00+00:00,Practice,Day 1,2026
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
668,NaN,1,38.0,True,False,False,None,1285,11291,NaN,2026-06-01 14:02:14.435617+00:00,0.0,2026-05-24 20:00:00+00:00,Race,Race,2026
669,NaN,63,29.0,True,False,False,None,1285,11291,NaN,2026-06-01 14:02:14.435617+00:00,0.0,2026-05-24 20:00:00+00:00,Race,Race,2026
670,NaN,14,23.0,True,False,False,None,1285,11291,NaN,2026-06-01 14:02:14.435617+00:00,0.0,2026-05-24 20:00:00+00:00,Race,Race,2026
671,NaN,23,11.0,True,False,False,None,1285,11291,NaN,2026-06-01 14:02:14.435617+00:00,0.0,2026-05-24 20:00:00+00:00,Race,Race,2026


In [48]:
buf = BytesIO()
df_sr.to_parquet(buf, index=False)
buf.seek(0)
payload = buf.getvalue()


clients = storage.Client()
bucket = clients.bucket(GCS_BUCKET)
blob = bucket.blob(path)

ArrowInvalid: ('Could not convert [79.507, 78.934, 78.518] with type list: tried to convert to double', 'Conversion failed for column duration with type object')